# Reference-data generation (pyscf / gpu4pyscf)

Folder-staged pipeline: each stage cell drains its input folder into the next, so a
molecule advances simply by running cells top-to-bottom. Stages are **idempotent and
resumable** — re-running a cell skips items already produced, so a Colab reconnect just
means re-running the cells.

```
00_inbox  ->  01_optimized  ->  02_frequencies  ->  03_wigner  ->  04_labeled
```

Per molecule: optimize (wB97M-V/def2-svpd) → harmonic frequencies → Wigner sampling
(300 K, 500 structures) → label each with energy/forces/dipole/quadrupole/
polarizability/dipole-derivatives → extended-XYZ matching `data/labels/*.extxyz`.

The compute core prefers **gpu4pyscf** (Colab GPU) and falls back to **pyscf** (local CPU).

## 1. Install
On Colab (GPU) this installs the gpu4pyscf stack; locally it installs the CPU pyscf backend.
Either way it then installs the `rsfff` package in editable mode.

In [ ]:
import sys, subprocess, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # GPU
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "pyscf==2.8.0", "basis-set-exchange==0.11", "geometric==1.1.0",
        "cupy-cuda12x==13.4.1", "cutensor-cu12==2.2.0",
        "gpu4pyscf-cuda12x==1.7.6"], check=True)
    # Clone the repo if it is not already present, then editable-install it.
    if not os.path.isdir("rsfff"):
        subprocess.run(["git", "clone", "https://github.com/heindelj/rsfff.git"], check=False)
    REPO = "rsfff" if os.path.isdir("rsfff/src") else "."
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO], check=True)
else:
    # Local CPU
    subprocess.run([sys.executable, "-m", "pip", "install", "pyscf>=2.14", "geometric"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".."], check=True)

print("install cell done; IN_COLAB =", IN_COLAB)

## 2. Setup
Pick the pipeline root (Colab Drive folder or a local directory), create the stage folders,
and report which backend is active.

In [ ]:
import os
# macOS libomp guard (harmless elsewhere); must be set before importing pyscf.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path
from rsfff.qcgen import pipeline as pl
from rsfff.qcgen.backend import backend_name

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/QC_runs")
else:
    # Local: keep runs under the repo's data/ tree.
    ROOT = Path("..") / "data" / "qc_runs"

cfg = pl.PipelineConfig(
    root=ROOT,
    xc="wB97M-V",
    basis="def2-svpd",
    temperature=300.0,
    n_samples=500,
    seed=0,
)
pl.make_dirs(cfg)
print("backend :", backend_name())
print("root    :", cfg.root.resolve())
print("method  :", cfg.xc, "/", cfg.basis, "| T =", cfg.temperature, "K | N =", cfg.n_samples)

## 3. Seed the inbox
Copy the molecules you want to process into `00_inbox/`. Charges/multiplicities are looked up
in `pipeline.SPECIES` (open-shell allowed, e.g. `oh_radical` → doublet). Edit `NAMES` to choose
which species to run; `None` seeds everything in `data/mols/`.

In [ ]:
MOLS = Path("..") / "data" / "mols"  # source geometries in the repo

NAMES = ["h2o"]  # e.g. ["h2o", "oh_radical"]; None = every *.xyz in data/mols

if NAMES is None:
    xyz_paths = sorted(MOLS.glob("*.xyz"))
else:
    xyz_paths = [MOLS / f"{n}.xyz" for n in NAMES]

seeded = pl.seed_inbox(cfg, xyz_paths)
print(f"seeded {len(seeded)} file(s) into {cfg.root / pl.INBOX}:")
print(" ", ", ".join(seeded) if seeded else "(nothing new; inbox already populated)")

## 4. Stage: optimize geometries
`00_inbox/*.xyz` → `01_optimized/<name>.xyz` (+ `<name>.json` meta).

In [ ]:
pl.run_stage(pl.STAGE_OPTIMIZE, cfg)

## 5. Stage: harmonic frequencies
`01_optimized/` → `02_frequencies/<name>.npz` (Hessian, masses, geometry). Analytic Hessian
where supported; automatic finite-difference fallback for UKS + NLC (open-shell wB97M-V).

In [ ]:
pl.run_stage(pl.STAGE_FREQUENCIES, cfg)

## 6. Stage: Wigner sampling
`02_frequencies/` → `03_wigner/<name>.extxyz` (`n_samples` geometries at `temperature`).

In [ ]:
pl.run_stage(pl.STAGE_WIGNER, cfg)

## 7. Stage: label properties
`03_wigner/` → `04_labeled/<name>.extxyz` (energy, forces, dipole, quadrupole,
polarizability, dipole derivatives). On success the original inbox `.xyz` is archived to
`_completed/`. This is the expensive stage; it streams frames to disk and writes atomically.

In [ ]:
pl.run_stage(pl.STAGE_LABEL, cfg)

## 8. Collect + summary
Concatenate all labeled species into one dataset file and report per-species frame counts and
any failures (tracebacks live in `_failed/`).

In [ ]:
out_path, counts = pl.collect(cfg)
print("combined dataset ->", out_path.resolve())
for name, n in sorted(counts.items()):
    print(f"  {name:14s} {n} frames")

failed = sorted((cfg.root / pl.FAILED).glob("*.error.log"))
if failed:
    print("\nfailures:")
    for f in failed:
        print("  ", f.name)